In [43]:
import json
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "amici2014calculated")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "amici2014calculated_standardized_NEW.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [44]:
import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1, sep=";")

In [45]:
# df['study_id']="amici2014calculated"
# df['experiment_name']="reciprocity task"

# df.columns = map(str.lower, df.columns)
# df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df = df.rename(columns={"species": "species_original",
#     "ab_subject": "ape",
#     "c_partner": "ape_2"})
# df.columns
# df['ape'] = df['ape'].str.rstrip()
# df['ape_2'] = df['ape_2'].str.rstrip()

# comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

# df_name  = pd.read_csv(comp_path_name_errors)
# for x,y in zip(df_name['wrong'],df_name['right']):
#     df['ape'].replace(x, y, inplace=True)
#     df['ape_2'].replace(x, y, inplace=True)

# df['dyad']=df.ape.str.cat(df.ape_2, sep='_')
# df.columns

# df['ape'].replace('', np.nan, inplace=True)
# df.dropna(subset=['ape'], inplace=True)

# role=[]
# role_2=[]
# for index, row in df.iterrows():
#     if not pd.isna(row['ape']):
#         role.append("subject_bc")
#     else:
#         role.append("")
# df = df.assign(role=role)
# for index, row in df.iterrows(): 
#     if not pd.isna(row['ape_2']):
#         role_2.append("partner_a")
#     else:
#         role_2.append("")
# df = df.assign(role_2=role_2)

In [46]:
df.rename(columns={"a_baited": "a_baited_subject_bc_acts",
                   "b_baited":"b_baited_partner_a_acts"}, inplace=True)
# df.columns

In [47]:
con_df = df[df.condition.str.contains("control")] 
con_a_df = con_df.drop(columns=['b_baited_partner_a_acts', 'participant_2','sex_2', 'role_2'])
con_b_df = con_df.drop(columns=['a_baited_subject_bc_acts', 'participant','sex', 'role'])


In [48]:
exp_df = df[df.condition.str.contains("experimental")]
exp_a_df = exp_df.drop(columns=['b_baited_partner_a_acts'])

exp_b_df = exp_df.drop(columns=['a_baited_subject_bc_acts'])

trial_update = [[exp_a_df, 4,7],
               [exp_a_df, 5,8],
                [exp_a_df, 6,9],
                [exp_b_df, 4,10],
                [exp_b_df, 5,11],
                [exp_b_df, 6,12],
                [exp_b_df, 1,4],
                [exp_b_df, 2,5],
                [exp_b_df, 3,6],]
for x,y,z in trial_update:
    x['trial'].replace(y, z, inplace=True)

In [49]:
data_frames = [con_a_df, con_b_df, exp_a_df, exp_b_df]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

# fulldf['trial']=fulldf['trial'].astype(int)
# fulldf=fulldf.sort_values(by = ['condition', 'trial'])

In [50]:
fulldf['participant_2'].replace('.', 'viringika', inplace=True)
fulldf['participant_2'].unique()

fulldf.rename(columns={"a_baited_subject_bc_acts": "a_baited_focal_participant_bc_acts"}, inplace=True)
fulldf['role'].replace('subject_bc', 'focal_participant_bc', inplace=True)

fulldf['experiment_name'].replace(' ', '_', inplace=True, regex=True)

In [51]:
amici2014calculated_standardized=fulldf[['study_id', 'experiment_name','participant',
        'sex','role', 'participant_2','sex_2', 'role_2','species', 'dyad',   'session',
       'trial', 'condition', 'a_baited_focal_participant_bc_acts', 'b_baited_partner_a_acts']]

In [52]:
comp_out_path_stand = os.path.join(out_pathway, 'amici2014calculated_standardized.csv')
amici2014calculated_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


In [53]:
names =amici2014calculated_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
amici2014calculated_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'amici2014calculated_glossary.csv')
amici2014calculated_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)